In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q mne scikit-learn matplotlib pandas numpy torch seaborn

import os
import gc
import random
import warnings

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, f1_score, roc_auc_score, confusion_matrix, precision_recall_curve, auc

# Tắt các cảnh báo hệ thống để log chạy sạch sẽ
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
mne.set_log_level("ERROR")

print("Đã chuẩn bị xong môi trường và các thư viện!")

In [ ]:
# Đường dẫn trực tiếp tới thư mục dữ liệu đã được Kaggle giải nén sẵn
input_dir = "/kaggle/input/datasets/ledaihoangnguyentmu/sleep-edf-database-expanded-1-0-0-zip/sleep-edf-database-expanded-1.0.0"

# Kiểm tra xem đường dẫn có hoạt động chính xác không
if os.path.exists(input_dir):
    print("✅ Đã kết nối thành công tới bộ dữ liệu Sleep-EDF!")
    print("Các thư mục con sẵn sàng sử dụng:")
    
    # Liệt kê các file/thư mục bên trong để chắc chắn
    files = os.listdir(input_dir)
    for f in files:
        print(f" - {f}")
else:
    print("⚠️ Không tìm thấy thư mục dữ liệu, vui lòng kiểm tra lại đường dẫn.")

cassette_dfs = pd.read_csv(os.path.join(input_dir, "cassette.csv"))
telemetry_dfs = pd.read_csv(os.path.join(input_dir, "telemetry.csv"))
print("✅ Đã tải dữ liệu từ cassette.csv và telemetry.csv vào DataFrame!")

In [ ]:
# Cấu hình tập trung - Bạn có thể dễ dàng sửa các tham số này tại đây
SEED = 42
HIDDEN_SIZE = 512    # Rất tốt, tăng sức mạnh ghi nhớ cho LSTM
NUM_LAYERS = 2       # SỬA: Đưa về 2 (hoặc 3) để mô hình hội tụ tốt, không bị lỗi đạo hàm
DROPOUT = 0.2        # Rất tốt, giúp mô hình dễ học hơn 0.3 cũ
KFOLD = 5           # SỬA: Tăng lên 5 fold để đánh giá mô hình chính xác hơn, tránh overfitting

WINDOW_SIZE = 5      # SỬA: Số lẻ, gọn gàng (nhìn 2 epoch trước, 2 epoch sau = 2.5 phút)
CENTER_IDX = 2       # SỬA: Tâm của cửa sổ 5 là 2. Công thức: (5 - 1) / 2 = 2

BATCH_SIZE = 32      # Okie, nếu máy báo lỗi Out of Memory thì hạ xuống 32 nhé
LEARNING_RATE = 1e-3 # Tạm ổn, nếu chạy mà thấy Loss trồi sụt thì hạ xuống 5e-4
MAX_EPOCHS = 100     # Rất tốt, cho mô hình cơ hội học lâu hơn
PATIENCE = 10         # GỢI Ý: Nên tăng từ 5 lên 7 hoặc 10, vì MAX_EPOCHS tận 100,
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = ['Wake', 'N1', 'N2', 'N3', 'REM']
VALID_STAGES = ['Sleep stage W', 'Sleep stage 1', 'Sleep stage 2', 'Sleep stage 3', 'Sleep stage 4', 'Sleep stage R']
MAPPING_STAGES = {'Sleep stage W': 0, 'Sleep stage 1': 1, 'Sleep stage 2': 2, 'Sleep stage 3': 3, 'Sleep stage 4': 3, 'Sleep stage R': 4}

print(f"🚀 Đang chạy trên thiết bị: {DEVICE}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

In [ ]:
class EMAEarlyStopping:
    """
    Tránh việc mô hình dừng quá sớm do sự biến động (oscillation) điểm số
    của WeightedRandomSampler bằng cách làm mượt trend qua hàm Exponential Moving Average.
    """
    def __init__(self, patience=5, alpha=0.15):
        self.patience = patience
        self.alpha = alpha
        self.best_ema_score = -float('inf')
        self.ema_score = None
        self.patience_counter = 0
        self.best_model_state = None

    def step(self, current_score, model):
        # Tính toán điểm mượt EMA
        if self.ema_score is None:
            self.ema_score = current_score
        else:
            self.ema_score = self.alpha * current_score + (1 - self.alpha) * self.ema_score

        # Kiểm tra cải tiến dựa trên điểm EMA
        if self.ema_score > self.best_ema_score:
            self.best_ema_score = self.ema_score
            self.patience_counter = 0
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False  # Chưa dừng
        else:
            self.patience_counter += 1
            if self.patience_counter >= self.patience:
                return True  # Kích hoạt dừng sớm
            return False

    def restore_best(self, model):
        if self.best_model_state is not None:
            model.load_state_dict(self.best_model_state)

In [ ]:
class SimpleANN(nn.Module):
    def __init__(self, input_size, num_classes=5, center_idx=15):
        super().__init__()
        self.center_idx = center_idx
        self.net = nn.Sequential(
            nn.Linear(input_size, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.LayerNorm(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)  # Không thêm Softmax ở đây, giữ nguyên Logits thô!
        )
    def forward(self, x):
        if x.dim() == 3: x = x[:, self.center_idx, :]
        return self.net(x)

class BiLSTMSleepStager(nn.Module):
    def __init__(self, input_size, hidden_size=256, num_layers=2, num_classes=5, dropout=0.3, center_idx=15):
        super().__init__()
        self.center_idx = center_idx
        self.input_norm = nn.LayerNorm(input_size)
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,
                            batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.attn_q = nn.Linear(hidden_size * 2, 1, bias=False)
        self.out_norm = nn.LayerNorm(hidden_size * 4)
        self.classifier = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(hidden_size * 4, num_classes))
        self._init_weights()

    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if "weight_ih" in name: nn.init.xavier_uniform_(param.data)
            elif "weight_hh" in name: nn.init.orthogonal_(param.data)
            elif "bias" in name: param.data.fill_(0.0); param.data[param.size(0)//4 : param.size(0)//2].fill_(1.0)
        nn.init.xavier_uniform_(self.attn_q.weight)

    def forward(self, x):
        x = self.input_norm(x)
        lstm_out, _ = self.lstm(x)
        h_centre = lstm_out[:, self.center_idx, :]
        attn_weights = F.softmax(self.attn_q(lstm_out).squeeze(-1), dim=1)
        h_context = (attn_weights.unsqueeze(-1) * lstm_out).sum(dim=1)
        return self.classifier(self.out_norm(torch.cat([h_centre, h_context], dim=1)))

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.smoothing = smoothing
        # KHÔNG sử dụng alpha (class_weights) ở đây nữa vì DataLoader đã dùng Sampler rồi
        self.register_buffer("alpha", None) 

    def forward(self, logits, targets):
        log_prob = F.log_softmax(logits, dim=-1)
        prob = torch.exp(log_prob)
        B, C = logits.shape

        # Label smoothing (Làm mượt nhãn nhiễu vùng biên)
        smooth_targets = torch.full_like(logits, self.smoothing / (C - 1))
        smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        # Tính toán Focal Loss nguyên bản (Chỉ tập trung đào sâu vào các mẫu khó học)
        focal_weight = torch.pow(1.0 - prob, self.gamma)
        loss = - (smooth_targets * focal_weight * log_prob).sum(dim=-1)

        return loss.mean()
    
class SleepSequenceDataset(Dataset):
    def __init__(self, file_dfs, pipeline_scaler, window_size=31, center_idx=15):
        self.windows, self.labels = [], []
        feature_cols = [c for c in file_dfs[0].columns if c != 'label']

        for df in file_dfs:
            if len(df) < window_size: continue
            # Biến đổi đặc trưng qua pipeline cao cấp (Log1p + StandardScaler)
            features_scaled = pipeline_scaler.transform(df[feature_cols].values)
            labels = df['label'].values

            for centre in range(center_idx, len(df) - center_idx):
                self.windows.append(features_scaled[centre - center_idx : centre + center_idx + 1])
                self.labels.append(labels[centre])

        self.windows = np.stack(self.windows, axis=0).astype(np.float32) if self.windows else np.empty((0, window_size, len(feature_cols)))
        self.labels = np.array(self.labels, dtype=np.int64)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return torch.from_numpy(self.windows[idx]), torch.tensor(self.labels[idx], dtype=torch.long)

In [ ]:
def train_model(model, train_loader, val_loader):
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1, eta_min=1e-6)
    criterion = FocalLoss(alpha=None, gamma=2.0, smoothing=0.1)
    stopper = EMAEarlyStopping(patience=5, alpha=0.15)

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for x_b, y_b in train_loader:
            x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            criterion(model(x_b), y_b).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        # Đánh giá sau mỗi epoch để check Early Stopping
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for x_b, y_b in val_loader:
                val_preds.extend(model(x_b.to(DEVICE)).argmax(dim=1).cpu().tolist())
                val_targets.extend(y_b.tolist())

        epoch_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)
        current_lr = optimizer.param_groups[0]["lr"]

        # Cập nhật Scheduler sau mỗi epoch
        scheduler.step()

        # Đút điểm số vào bộ lọc EMA để kiểm tra dừng sớm
        if stopper.step(epoch_f1, model):
            print(f"   [Early Stopping] Dừng tại epoch {epoch} — Best EMA Val F1 = {stopper.best_ema_score:.4f}")
            break

    # Phục hồi lại trọng số tại thời điểm điểm EMA đạt đỉnh tốt nhất
    stopper.restore_best(model)

def evaluate_loader(model, loader):
    model.eval()
    all_trues = []
    all_preds = []
    all_probs = [] # Lưu xác suất để tính PR_AUC
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            
            # Lấy xác suất chuẩn hóa bằng Softmax
            probs = F.softmax(logits, dim=-1)
            preds = torch.argmax(logits, dim=-1)
            
            all_trues.extend(y.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    return np.array(all_trues), np.array(all_preds), np.array(all_probs)

def calculate_metrics(targets, preds, probs):
    return {
        'acc': accuracy_score(targets, preds),
        'precision': precision_score(targets, preds, average='macro', zero_division=0),
        'f1': f1_score(targets, preds, average='macro', zero_division=0),
        'roc_auc': roc_auc_score(targets, probs, multi_class='ovr')
    }

def calculate_metrics_v2(y_true, y_pred, y_probs):
    # Tính Acc, Precision, F1 thông thường
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    # Binarize nhãn để tính PR-AUC cho đa lớp (5 lớp)
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3, 4])

    # Tính PR-AUC Macro
    pr_auc_scores = []
    for i in range(5): 
        precision_curve, recall_curve, _ = precision_recall_curve(y_true_bin[:, i], y_probs[:, i])
        precision_recall_auc = auc(recall_curve, precision_curve)
        pr_auc_scores.append(precision_recall_auc)

    macro_pr_auc = np.mean(pr_auc_scores)

    return {
        'acc': acc,
        'precision': precision,
        'f1': f1,
        'pr_auc': macro_pr_auc
    }

In [1]:
skf = StratifiedKFold(n_splits=KFOLD, shuffle=True, random_state=SEED)
results = {'Simple ANN': [], 'BiLSTM': []}
final_evaluation_data = {'Simple ANN': {'true': [], 'pred': []}, 'BiLSTM': {'true': [], 'pred': []}}
file_majority_labels = [int(df['label'].mode().iloc[0]) for df in cassette_dfs]

for fold, (train_idx, val_idx) in enumerate(skf.split(cassette_dfs, file_majority_labels)):
    print(f"\n╔════════════════════════════════════╗\n║             FOLD {fold+1} / {KFOLD}             ║\n╚════════════════════════════════════╝")
    train_dfs = [cassette_dfs[i] for i in train_idx]
    val_dfs = [cassette_dfs[i] for i in val_idx]

    feature_cols = [c for c in train_dfs[0].columns if c != 'label']
    raw_train_matrix = pd.concat(train_dfs, ignore_index=True)[feature_cols].values

    pipeline_scaler = Pipeline([
        ('log1p', FunctionTransformer(np.log1p, validate=True)),
        ('scaler', StandardScaler())
    ])
    pipeline_scaler.fit(raw_train_matrix)

    # Đưa pipeline đã fit vào khởi tạo Dataset
    train_set = SleepSequenceDataset(train_dfs, pipeline_scaler, WINDOW_SIZE, CENTER_IDX)
    val_set = SleepSequenceDataset(val_dfs, pipeline_scaler, WINDOW_SIZE, CENTER_IDX)
    test_set = SleepSequenceDataset(telemetry_dfs, pipeline_scaler, WINDOW_SIZE, CENTER_IDX)

    class_weights = compute_class_weight("balanced", classes=np.array([0,1,2,3,4]), y=train_set.labels)
    sampler = WeightedRandomSampler(weights=torch.tensor(class_weights[train_set.labels], dtype=torch.float32), num_samples=len(train_set), replacement=True)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # 1. Chạy Simple ANN
    print(" -> Đang huấn luyện Simple ANN...")
    ann = SimpleANN(input_size=len(feature_cols), center_idx=CENTER_IDX).to(DEVICE)
    train_model(ann, train_loader, val_loader)
    t_ann, p_ann, pr_ann = evaluate_loader(ann, test_loader)
    results['Simple ANN'].append(calculate_metrics_v2(t_ann, p_ann, pr_ann))
    final_evaluation_data['Simple ANN']['true'].extend(t_ann)
    final_evaluation_data['Simple ANN']['pred'].extend(p_ann)

    # 2. Chạy BiLSTM + Attention
    print(" -> Đang huấn luyện BiLSTM + Attention...")
    lstm = BiLSTMSleepStager(input_size=len(feature_cols), hidden_size= HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT, center_idx=CENTER_IDX).to(DEVICE)
    train_model(lstm, train_loader, val_loader)
    t_lstm, p_lstm, pr_lstm = evaluate_loader(lstm, test_loader)
    results['BiLSTM'].append(calculate_metrics_v2(t_lstm, p_lstm, pr_lstm))
    final_evaluation_data['BiLSTM']['true'].extend(t_lstm)
    final_evaluation_data['BiLSTM']['pred'].extend(p_lstm)
    
    print(f"   ↳ [DEBUG] Kết quả Fold {fold+1} - Simple ANN: {results['Simple ANN'][-1]}")
    print(f"   ↳ [DEBUG] Kết quả Fold {fold+1} - BiLSTM: {results['BiLSTM'][-1]}")

    del ann, lstm; torch.cuda.empty_cache(); gc.collect()

print("\n🎉 Toàn bộ vòng lặp thực nghiệm nâng cao đã hoàn thành xuất sắc!")

NameError: name 'StratifiedKFold' is not defined

In [ ]:
print("\n📈 CHI TIẾT KẾT QUẢ QUA CÁC FOLDS:")

fold_records = []
for model_name, folds_list in results.items():
    for fold_idx, metrics in enumerate(folds_list):
        rec = metrics.copy()
        rec['Model'] = model_name
        rec['Fold'] = f"Fold {fold_idx + 1}"
        fold_records.append(rec)

df_folds = pd.DataFrame(fold_records)

# Hiển thị bảng chi tiết từng fold
for model_name in results.keys():
    print(f"\n➔ Model: {model_name}")
    df_model_folds = df_folds[df_folds['Model'] == model_name].set_index('Fold')
    display(df_model_folds[['acc', 'precision', 'f1', 'pr_auc']].round(4))

# Hiển thị bảng trung bình
print("\n📊 AVERAGE EVALUATION RESULTS ON INDEPENDENT TEST SET (TELEMETRY):")
df_summary = df_folds.groupby('Model')[['acc', 'precision', 'f1', 'pr_auc']].mean()
display(df_summary.round(4))


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
metrics_to_plot = ['acc', 'precision', 'f1']
existing_folds = df_folds['Fold'].unique()

# Đồ thị 1: So sánh Acc, Precision, F1
for model_name in results.keys():
    df_m = df_folds[df_folds['Model'] == model_name]
    for metric in metrics_to_plot:
        axes[0].plot(
            df_m['Fold'], df_m[metric], 
            marker='o', linestyle='-', label=f"{model_name} - {metric.upper()}",
            alpha=0.8
        )

axes[0].set_title("Performance Metrics across Folds", fontweight='bold', fontsize=12)
axes[0].set_ylabel("Score (0.0 - 1.0)")
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='lower left', bbox_to_anchor=(0, -0.32), ncol=2)


# --- 3. VẼ BIỂU ĐỒ ĐƯỜNG RIÊNG CHO PR_AUC ---
for model_name in results.keys():
    df_m = df_folds[df_folds['Model'] == model_name]
    axes[1].plot(
        df_m['Fold'], df_m['pr_auc'], 
        marker='s', markersize=8, linestyle='--', linewidth=2,
        label=f"{model_name} - PR_AUC"
    )

axes[1].set_title("Target Metric (PR_AUC) across Folds", fontweight='bold', fontsize=12)
axes[1].set_ylabel("AUC Score")
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='lower right', bbox_to_anchor=(1, -0.22))

plt.tight_layout()
plt.show()


# --- 4. VẼ CONFUSION MATRIX CHUẨN (Khắc phục hoàn toàn lỗi trích xuất mẫu) ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

num_actual_folds = len(existing_folds)

for idx, model_name in enumerate(['Simple ANN', 'BiLSTM']):
    # Tìm fold có F1-score cao nhất của model này
    df_m = df_folds[df_folds['Model'] == model_name]
    best_fold_idx = df_m['f1'].idxmax()
    best_fold_num = df_m.loc[best_fold_idx, 'Fold']
    
    # Tính chính xác kích thước tập test của 1 fold đơn lẻ
    total_test_samples = len(final_evaluation_data[model_name]['true']) // num_actual_folds
    
    fold_number = int(best_fold_num.split()[-1]) - 1 # Chuyển đổi chữ "Fold X" thành chỉ số index
    start_pos = fold_number * total_test_samples
    end_pos = start_pos + total_test_samples
    
    # Trích xuất chuẩn xác phân đoạn dữ liệu của fold tốt nhất
    y_true_best = final_evaluation_data[model_name]['true'][start_pos:end_pos]
    y_pred_best = final_evaluation_data[model_name]['pred'][start_pos:end_pos]

    # Tính toán Confusion Matrix
    cm = confusion_matrix(y_true_best, y_pred_best, labels=[0, 1, 2, 3, 4])
    
    # Chuẩn hóa theo hàng dòng (True labels), phòng tránh lỗi chia cho 0 bằng giá trị eps nhỏ
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)

    sns.heatmap(
        cm_norm, annot=True, fmt=".2f", cmap="Purples",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=axes[idx], cbar=False
    )

    axes[idx].set_title(
        f"CM - {model_name} ({best_fold_num} - Best F1)", 
        fontweight='bold', fontsize=12
    )
    axes[idx].set_xlabel("Predicted Label")
    axes[idx].set_ylabel("True Label")

plt.tight_layout()
plt.show()